In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HOME"] = "/workspace/hf-cache"
os.environ["DATASET_REPO_ID"] = "HCHoongChing/bimanual-piper-dataset-001"
os.environ["OPENPI_DATA_HOME"] = "/workspace/openpi-cache"

# openpi dependencies

In [ ]:
! pip install uv
! GIT_LFS_SKIP_SMUDGE=1 uv sync
! GIT_LFS_SKIP_SMUDGE=1 uv pip install -e .

In [ ]:
! apt update
! apt install -y ffmpeg libavcodec-dev libavformat-dev libavutil-dev

In [ ]:
! pip install -U "huggingface_hub[cli,hf_transfer]"

In [ ]:
from huggingface_hub import snapshot_download

# Download dataset

In [ ]:
print(os.environ.get("DATASET_REPO_ID"))
while True:
    try:
        snapshot_download(
            repo_id="griffinlabs/new_v21_B001_to_B005_single_arm_hqc_data", local_dir="/workspace/dataset", repo_type="dataset"
        )
        break
    except Exception as e:
        print(f"Error: {e}, retrying...")

Download Base pytorch (if needed)

In [ ]:
# download pytorch_base
while True:
    try:
        snapshot_download(
            repo_id="griffinlabs/pytorch-pi05-base", local_dir="/workspace/checkpoint/pi05_base_pytorch", repo_type="model"
        )
        break
    except Exception as e:
        print(f"Error: {e}, retrying...")

# Proprocess dataset

In [ ]:
! uv run scripts/compute_norm_stats.py --config-name pi05_tcr_full_finetune

# Train!

In [ ]:
import sys
sys.stdout = open("log.txt", "a")

In [ ]:
#single gpu
! uv pip show transformers
! cd /workspace/openpi && cp -r src/openpi/models_pytorch/transformers_replace/* .venv/lib/python3.11/site-packages/transformers/ 
! cd /workspace/openpi && uv run scripts/train_pytorch.py pi05_tcr_full_finetune_pytorch \
--exp_name pi05_412ep \
--checkpoint-base-dir /workspace/checkpoints \
--wandb-enabled --overwrite \
--data.repo-id /workspace/dataset \
--batch-size 32


In [ ]:
# multigpu
! uv pip show transformers
! cp -r ./src/openpi/models_pytorch/transformers_replace/* .venv/lib/python3.11/site-packages/transformers/
! uv run torchrun --standalone --nnodes=1 --nproc_per_node=4 scripts/train_pytorch.py pi05_tcr_full_finetune_pytorch \
--exp_name pi05_B001_B005 \
--checkpoint-base-dir /workspace/checkpoints \
--data.repo-id /workspace/dataset2 \
--pytorch_weight_path /workspace/pi05_base_pytorch \
--wandb-enabled \
--batch-size 256
